Обучите простую рекуррентную нейронную сеть (без GRU/LSTM, без внимания) решать задачу дешифровки шифра Цезаря:
1. Написать алгоритм шифра Цезаря для генерации выборки (сдвиг на N каждой буквы). Например если N=2, то буква A переходит в букву C. Можно поиграться с языком на выбор (немецкий, русский и т.д.)
2. Создать архитектуру рекуррентной нейронной сети.
3. Обучить ее (вход - зашифрованная фраза, выход - дешифрованная фраза).
4. Проверить качество модели.	

In [1]:
import pandas as pd
import time
import torch
import random

In [2]:
# предварительно сгенерированные фразы в виде «цитатника» из разных философских систем
def load_from_file(filename="./input/task_1_input.txt"):
    with open(filename, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]
    return lines

In [3]:
original_texts = load_from_file()
random.shuffle(original_texts) # перемешиваем строки

In [4]:
alphabetLower = "абвгдеёжзийклмнопрстуфхцчшщъыьэюя"
alphabetUpper = "АБВГДЕЁЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯ"

In [5]:
# определим алгоритм шифра Цезаря со сдвигом для каждого символа
def caesar_encrypt(text, shift):    
    result = ""
    for char in text:
        lower_char = char.lower()
        if lower_char in alphabetLower:
            # находим новый индекс с учетом сдвига и размера алфавита
            idx = alphabetLower.index(lower_char)
            new_idx = (idx + shift) % 33
            new_char = alphabetLower[new_idx]
            # возвращаем исходный регистр
            result += new_char.upper() if char.isupper() else new_char
        else:
            # пробелы, знаки препинания и т.д. оставляем as is
            result += char
    return result

In [6]:
caesar_encrypt('Смысл жизни не в том, чтобы найти себя, а в том, Чтобы создать себя', 3)

'Фпюфо йлкрл рз е хсп, ъхсдю ргмхл фздв, г е хсп, Ъхсдю фскжгхя фздв'

In [7]:
# исходные символы текстовых фрагментов
orig_fragments = [[c for c in ot] for ot in original_texts if type(ot) is str]
# зашифрованные символы (шифром Цезаря со смещением 3) текстовых фрагментов
encrypted_fragments = [[c for c in caesar_encrypt(ot, 3)] for ot in original_texts if type(ot) is str]

Разделим текстовые фрагменты на тренировочный и тестовый наборы

In [8]:
# разделяем 85% на обучение, 15% на тест
split_idx = int(len(orig_fragments) * 0.85)

train_orig_fragments = orig_fragments[:split_idx]
test_orig_fragments = orig_fragments[split_idx:]

train_encr_fragments = encrypted_fragments[:split_idx]
test_encr_fragments = encrypted_fragments[split_idx:]

Сделаем словарь символов текста

In [9]:
CHARS = set(alphabetLower + alphabetUpper + '—, ')
INDEX_TO_CHAR = ['none'] + [w for w in CHARS]
CHAR_TO_INDEX = {w: i for i, w in enumerate(INDEX_TO_CHAR)}

In [10]:
MAX_LEN = 80
X_orig = torch.zeros((len(train_orig_fragments), MAX_LEN), dtype=int)
X_encrypt = torch.zeros((len(train_encr_fragments), MAX_LEN), dtype=int)

for i in range(len(train_orig_fragments)):
    for j, w in enumerate(train_orig_fragments[i]):
        if j >= MAX_LEN:
            break
        X_orig[i, j] = CHAR_TO_INDEX.get(w, CHAR_TO_INDEX['none'])
        
for i in range(len(train_encr_fragments)):
    for j, w in enumerate(train_encr_fragments[i]):
        if j >= MAX_LEN:
            break
        X_encrypt[i, j] = CHAR_TO_INDEX.get(w, CHAR_TO_INDEX['none'])      

Создадим модель нейронной сети

In [11]:
class Network(torch.nn.Module):
    def __init__(self):
        super(Network, self).__init__()
        # cоздадим эмбеддинги на 20 элементов для входных текстов
        self.embed = torch.nn.Embedding(len(CHAR_TO_INDEX), 20)
        # cоздадим слой RNN со скрытыми состояниями на 128 элементов
        self.rnn = torch.nn.RNN(20, 128, batch_first=True)
        # cоздадим полносвязный слой RNN из скрытого состояния rnn в символ (70)
        self.linear = torch.nn.Linear(128, len(INDEX_TO_CHAR))
        
    def forward(self, sentences, state=None):
        embed = self.embed(sentences)
        o, a = self.rnn(embed)
        out = self.linear(o)  
        return out

In [12]:
model = Network()

In [13]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=.05)

In [14]:
for ep in range(300):
    start = time.time()
    train_loss = 0.
    train_passed = 0

    batch_size = 30
    for i in range(int(len(X_encrypt) / batch_size)):
        # для обучения передаем на вход зашифрованный текст
        # X_batch представляет собой последовательности символов (индексов)
        X_batch = X_encrypt[i * batch_size:(i + 1) * batch_size]
        
        # Y_batch — это целевая метка из оригинального текста, с которой сравниваются предсказания модели.
        Y_batch = X_orig[i * batch_size:(i + 1) * batch_size].flatten()
        
        optimizer.zero_grad()
        answers = model.forward(X_batch)
        answers = answers.view(-1, len(INDEX_TO_CHAR))
        
        loss = criterion(answers, Y_batch)
        train_loss += loss.item()

        loss.backward()
        optimizer.step()
        train_passed += 1

    if (ep % 50 == 0):
        print("Epoch {}. Time: {:.3f}, Train loss: {:.3f}".format(ep, time.time() - start, train_loss / train_passed))

Epoch 0. Time: 1.365, Train loss: 3.667
Epoch 50. Time: 0.283, Train loss: 0.261
Epoch 100. Time: 1.423, Train loss: 0.120
Epoch 150. Time: 1.411, Train loss: 0.071
Epoch 200. Time: 1.573, Train loss: 0.048
Epoch 250. Time: 1.619, Train loss: 0.035


In [15]:
def generate_sentence(model, sentence, max_len=80):
    # Преобразуем предложение в индексы
    x = torch.zeros((1, len(sentence)), dtype=int)
    for j, w in enumerate(sentence):
        if j >= max_len:
            break
        x[0, j] = CHAR_TO_INDEX.get(w, CHAR_TO_INDEX['none'])

    # расшифровка текста
    generated_sentence = list()

    # получаем логиты предсказания/расшифровки
    o = model(x)
    # выход модели для всех символов
    output = o[0] 
    
    # генерация символов
    for i in range(len(sentence)):
        last_output = output[i]
        
        # находим индекс самой вероятной буквы
        predicted_index = torch.argmax(last_output).item()
        
        # получаем символ, соответствующий индексу
        predicted_char = INDEX_TO_CHAR[predicted_index]

        # добавляем предсказанную букву в предложение
        if predicted_char == 'none':
            # для выделения позиции символа, отсутствующего в словаре
            generated_sentence.append('█')
        else:
            generated_sentence.append(predicted_char)

    return ''.join(generated_sentence)

In [16]:
import numpy as np

def estimate_accuracy(encr_fragments, orig_fragments):
    """
        Определяет accuracy расшифрованных текстовых фрагментов
    """ 
    decoded_fragments = [generate_sentence(model, enc_fr) for enc_fr in encr_fragments]
    joined_orig_fragments = [''.join(orig_fr) for orig_fr in orig_fragments]  
    mask = [orig == dec for orig, dec in zip(joined_orig_fragments, decoded_fragments)]
    return np.mean(mask)      

In [17]:
train_accuracy = estimate_accuracy(train_encr_fragments, train_orig_fragments)
print(f"Accuracy для тренировочных данных: {train_accuracy:.2f}")

Accuracy для тренировочных данных: 0.82


In [26]:
test_accuracy = estimate_accuracy(test_encr_fragments, test_orig_fragments)
print(f"Accuracy для тестовых данных: {test_accuracy:.2f}")

Accuracy для тестовых данных: 0.75


In [19]:
# используем сходство строк по Левенштейну
import Levenshtein

Оценим качество модели по трем вариантам одной фразы

In [20]:
input_test_text = 'явленная истина — это лишь тишина, застывшая между буквами, словами и смыслами'
decoded = generate_sentence(model, sentence = caesar_encrypt(input_test_text, 3))    
decoded

'явленная истина — это лишь тишина, застывшая между буквами, словами и смыслами'

In [21]:
Levenshtein.distance(decoded, input_test_text)

0

Дешифрованная фраза совпадает с исходной фразой \
в случае исходной фразы в нижнем регистре.

In [22]:
input_test_text = 'Явленная истина — это лишь тишина, Застывшая меЖду Буквами, словами и сМыслами'
decoded = generate_sentence(model, sentence = caesar_encrypt(input_test_text, 3))
decoded

'Явленная истина — это лишь тишина, Застывшая мезду Буквами, словами и сМыслами'

In [23]:
Levenshtein.distance(decoded, input_test_text)

1

Дешифрованная фраза отличается от исходной фразы на один буквенный символ (Ж - з) \
в случае исходной фразы со строчными и заглавными буквами.

In [24]:
input_test_text = 'Явленная истина — это Лишь тишина. Застывшая меЖду буквами, словами и смыслами'
decoded = generate_sentence(model, sentence = caesar_encrypt(input_test_text, 3)) 
decoded

'Явленная истина — это Лишь тишина█ Застывшая мезду буквами, словами и смыслами'

In [25]:
Levenshtein.distance(decoded, input_test_text)

2

Дешифрованная фраза отличается от исходной фразы на один буквенный символ (Ж - з) \
и символ-заполнитель для символа, не включенного в словарь\
в случае исходной фразы со строчными и заглавными буквами.